# **DLRM**

Deep Learning Recommendation Model- instead of just using a normal NN and expect it to understand recommendation attributes organically, DLRM uses dedicated specific neural pathways to dense and sparse features.Categorical features are mapped to dense vectors through embedding tables, while continuous features are transformed through a bottom MLP to yield a vector exact same dimensionality.

To capture structural relationships, DLRM explicitly computes second order interactions by taking dot product of all pairs of embedding vectors and the processed dense output. Finally these airwise interaction terms are concatenated with the original dense vector and passed through a top MLP to predict the target probability.

This architectural seperation was made to help data parallelism for compute-bound MLPs and model parallelism for memory-bound conditions.

COMPARISON W PREV TASKS' MODELS:

->Task1> MF is restricted strictly to User-Item relationships while DLRM generalizes this by taking dozens of contextual features (time, device, campaign) and treating their embeddings identically to latent user/item factors.

->Task2> Vanilla MLPs rely entirely on standard dense layers to implicitly guess feature interactions, thus requiring large parameter counts to achieve high expressivity. DLRM enforces an explicit mathematical dot-product interaction, significantly reducing the dimensionality and parameter burden of the top-level layers.

DRAWBACK/COMPLEXITY ADDED:
DLRM demands that all categorical embeddings and the bottom MLP output share the exact same dimensionality to allow for dot products. Furthermore, the interaction phase requires $O(N^2)$ dot products (where N is the number of features), introducing computational overhead as the feature space grows.

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class DLRM(nn.Module):
    def __init__(self, num_numeric, cat_dims, embed_dim=16, bottom_mlp_dims=[64, 16], top_mlp_dims=[128, 64]):
        super().__init__()
        
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=dim, embedding_dim=embed_dim) for dim in cat_dims
        ])
        
        # Bottom MLP for Dense Features
        layers = []
        in_dim = num_numeric
        for out_dim in bottom_mlp_dims:
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(nn.ReLU())
            in_dim = out_dim
        if in_dim != embed_dim:
            layers.append(nn.Linear(in_dim, embed_dim))
            layers.append(nn.ReLU())
        self.bottom_mlp = nn.Sequential(*layers)

        num_vectors = len(cat_dims) + 1
        num_interactions = (num_vectors * (num_vectors - 1)) // 2

        layers = []
        in_dim = num_interactions + embed_dim
        for out_dim in top_mlp_dims:
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=0.1))
            in_dim = out_dim
        layers.append(nn.Linear(in_dim, 1))
        self.top_mlp = nn.Sequential(*layers)

    def forward(self, numeric_x, cat_x):
        v_dense = self.bottom_mlp(numeric_x).unsqueeze(1) 
        v_cats = [emb(cat_x[:, i]).unsqueeze(1) for i, emb in enumerate(self.embeddings)]
        
        all_vectors = torch.cat([v_dense] + v_cats, dim=1) 

        batch_matrix_dots = torch.bmm(all_vectors, all_vectors.transpose(1, 2)) 
        num_vectors = all_vectors.size(1)
        triu_indices = torch.triu_indices(num_vectors, num_vectors, offset=1)
        interactions = batch_matrix_dots[:, triu_indices[0], triu_indices[1]] 
        
        top_input = torch.cat([v_dense.squeeze(1), interactions], dim=1) 
        return self.top_mlp(top_input).squeeze(-1)

In [ ]:
class DLRMAblation(DLRM):
    def __init__(self, num_numeric, cat_dims, embed_dim=16, bottom_mlp_dims=[64, 16], top_mlp_dims=[128, 64]):
        super().__init__(num_numeric, cat_dims, embed_dim, bottom_mlp_dims, top_mlp_dims)
        
        in_dim = (len(cat_dims) * embed_dim) + embed_dim
        layers = []
        for out_dim in top_mlp_dims:
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=0.1))
            in_dim = out_dim
        layers.append(nn.Linear(in_dim, 1))
        self.top_mlp = nn.Sequential(*layers)

    def forward(self, numeric_x, cat_x):
        v_dense = self.bottom_mlp(numeric_x)
        v_cats = [emb(cat_x[:, i]) for i, emb in enumerate(self.embeddings)]
        top_input = torch.cat([v_dense] + v_cats, dim=1) 
        return self.top_mlp(top_input).squeeze(-1)